# SAE Dual-Pipeline Visualization

这个 notebook 读取 `scripts/run_sae_pipeline.py` 生成的 `pipeline_summary.json`，对 `standard` 和 `attnres` 两个模型的 SAE 训练曲线与结构评估结果做并排可视化。

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    candidate = Path.cwd() if start is None else start
    for path in [candidate, *candidate.parents]:
        if (path / "stream_analysis").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
SUMMARY_PATH = REPO_ROOT / "outputs" / "sae_dual_pipeline" / "pipeline_summary.json"
REPO_ROOT, SUMMARY_PATH

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pipeline summary: {SUMMARY_PATH}\n"
        "先运行 accelerate launch scripts/run_sae_pipeline.py ... 生成双模型 SAE 结果。"
    )

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    pipeline_summary = json.load(handle)

overview_rows = []
for model_type, payload in pipeline_summary["models"].items():
    overview_rows.append(
        {
            "model_type": model_type,
            "checkpoint": payload["checkpoint_path"],
            "best_val_recon_mse": payload["best_val_recon_mse"],
            "sae_dir": payload["sae_dir"],
            "eval_summary_path": payload.get("eval_summary_path", ""),
        }
    )

if pd is not None:
    display(pd.DataFrame(overview_rows))
else:
    overview_rows

In [ ]:
def read_metrics_csv(path: str | Path):
    rows = []
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            rows.append({key: float(value) for key, value in row.items()})
    return rows


def load_eval_summary(path: str | Path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


train_curves = {
    model_type: read_metrics_csv(payload["metrics_path"])
    for model_type, payload in pipeline_summary["models"].items()
}
eval_payloads = {
    model_type: load_eval_summary(payload["eval_summary_path"])
    for model_type, payload in pipeline_summary["models"].items()
}

list(train_curves.keys())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for axis, metric_key, title in [
    (axes[0], "train_recon_mse", "Train Reconstruction MSE"),
    (axes[1], "val_recon_mse", "Validation Reconstruction MSE"),
]:
    for model_type, rows in train_curves.items():
        steps = [int(row["step"]) for row in rows]
        values = [row[metric_key] for row in rows]
        axis.plot(steps, values, marker="o", label=model_type)
    axis.set_title(title)
    axis.set_xlabel("step")
    axis.set_ylabel(metric_key)
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
metric_keys = ["recon_mse", "normalized_recon_mse", "avg_l0", "dead_latent_frac"]
model_order = list(pipeline_summary["models"].keys())
x = np.arange(len(metric_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
for idx, model_type in enumerate(model_order):
    summary = eval_payloads[model_type]["summary"]
    values = [float(summary[key]) for key in metric_keys]
    offset = (-0.5 + idx) * width
    ax.bar(x + offset, values, width=width, label=model_type)

ax.set_xticks(x)
ax.set_xticklabels(metric_keys, rotation=20)
ax.set_ylabel("value")
ax.set_title("SAE Evaluation Comparison")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(model_order), figsize=(7 * len(model_order), 5))
if len(model_order) == 1:
    axes = [axes]

for axis, model_type in zip(axes, model_order):
    decoder_overlap = eval_payloads[model_type]["decoder_overlap"].get("heatmap")
    if decoder_overlap is None:
        axis.axis("off")
        axis.set_title(f"{model_type}: decoder overlap heatmap unavailable")
        continue
    image = axis.imshow(np.asarray(decoder_overlap, dtype=float), aspect="auto", interpolation="nearest")
    axis.set_title(f"{model_type}: decoder overlap")
    axis.set_xlabel("latent j")
    axis.set_ylabel("latent i")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(model_order), figsize=(7 * len(model_order), 5))
if len(model_order) == 1:
    axes = [axes]

for axis, model_type in zip(axes, model_order):
    coactivation = eval_payloads[model_type]["coactivation"].get("heatmap")
    if coactivation is None:
        axis.axis("off")
        axis.set_title(f"{model_type}: coactivation heatmap unavailable")
        continue
    image = axis.imshow(np.asarray(coactivation, dtype=float), aspect="auto", interpolation="nearest")
    axis.set_title(f"{model_type}: coactivation")
    axis.set_xlabel("latent j")
    axis.set_ylabel("latent i")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()